# debt present value

## Setup

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parameter

In [ ]:
par = 1000000
coupon_rate = 3 / 100
months = 36 # maturity in months
freq = 4
ytm = 5.0 / 100
tax_rate = 15.0 / 100

## Cashflow

In [ ]:
def periods(months, freq):
    return np.arange(1, int(round(months * freq / 12)) + 1)


def cash_flows(par, coupon_rate, months, freq):
    coupon = par * coupon_rate / freq
    flows = np.full(len(periods(months, freq)), coupon)
    flows[-1] += par
    return flows


def after_tax_cash_flows(par, coupon_rate, months, freq, tax_rate):
    coupon = par * coupon_rate / freq
    return cash_flows(par, coupon_rate, months, freq) - coupon * tax_rate

## Discount value

In [ ]:
def discount_factors(ytm, months, freq):
    return (1 + ytm / freq) ** -periods(months, freq)

## Present value

In [ ]:
def present_value(par, coupon_rate, months, ytm, freq=1, tax_rate=0.0):
    flows = after_tax_cash_flows(par, coupon_rate, months, freq, tax_rate)
    factors = discount_factors(ytm, months, freq)
    return float(flows @ factors)

In [ ]:
price = present_value(par, coupon_rate, months, ytm, freq)
price_after_tax = present_value(par, coupon_rate, months, ytm, freq, tax_rate)
pv_tax = price - price_after_tax

price, price_after_tax, pv_tax

(472301.7200849078, 466069.607104012, 6232.112980895792)

## Schedule

In [ ]:
schedule = pd.DataFrame({
    "period": periods(months, freq),
    "cash_flow": cash_flows(par, coupon_rate, months, freq),
    "after_tax_cash_flow": after_tax_cash_flows(par, coupon_rate, months, freq, tax_rate),
    "discount_factor": discount_factors(ytm, months, freq),
})
schedule["present_value"] = schedule["cash_flow"] * schedule["discount_factor"]
schedule["after_tax_present_value"] = schedule["after_tax_cash_flow"] * schedule["discount_factor"]
schedule.round(2)

,period,cash_flow,after_tax_cash_flow,discount_factor,present_value,after_tax_present_value
0,1,3750.0,3187.5,0.99,3703.70,3148.15
1,2,3750.0,3187.5,0.98,3657.98,3109.28
2,3,3750.0,3187.5,0.96,3612.82,3070.90
3,4,3750.0,3187.5,0.95,3568.22,3032.98
4,5,3750.0,3187.5,0.94,3524.16,2995.54
5,6,3750.0,3187.5,0.93,3480.66,2958.56
6,7,3750.0,3187.5,0.92,3437.68,2922.03
7,8,3750.0,3187.5,0.91,3395.24,2885.96
8,9,3750.0,3187.5,0.89,3353.33,2850.33
9,10,3750.0,3187.5,0.88,3311.93,2815.14


## Price and yield

In [ ]:
curve = pd.DataFrame({"ytm": np.arange(0.02, 0.0801, 0.005)})
curve["price"] = [present_value(par, coupon_rate, months, y, freq) for y in curve["ytm"]]
curve["price_after_tax"] = [present_value(par, coupon_rate, months, y, freq, tax_rate) for y in curve["ytm"]]
curve.round({"ytm": 4, "price": 2, "price_after_tax": 2})

,ytm,price,price_after_tax
0,0.020,514523.67,507988.02
1,0.025,507204.00,500720.40
2,0.030,500000.00,493567.86
3,0.035,492909.70,486528.43
4,0.040,485931.15,479600.17
5,0.045,479062.45,472781.19
6,0.050,472301.72,466069.61
7,0.055,465647.12,459463.61
8,0.060,459096.86,452961.38
9,0.065,452649.14,446561.18


## IRR

In [ ]:
def net_present_value(price, par, coupon_rate, months, rate, freq=1, tax_rate=0.0):
    return present_value(par, coupon_rate, months, rate, freq, tax_rate) - price


def irr(price, par, coupon_rate, months, freq=1, tax_rate=0.0, tol=1e-12):
    low, high = -freq + 1e-9, 1.0
    while high - low > tol:
        mid = (low + high) / 2
        if net_present_value(price, par, coupon_rate, months, mid, freq, tax_rate) > 0:
            low = mid
        else:
            high = mid
    return (low + high) / 2

In [ ]:
market_price = 991000

irr_pre_tax = irr(market_price, par, coupon_rate, months, freq)
irr_after_tax = irr(market_price, par, coupon_rate, months, freq, tax_rate)

irr_pre_tax, irr_after_tax

(-0.2006223434758957, -0.2037675739878534)

In [ ]:
quotes = pd.DataFrame({"price": np.arange(850_000, 1_150_001, 50_000)})
quotes["irr"] = [irr(p, par, coupon_rate, months, freq) for p in quotes["price"]]
quotes["irr_after_tax"] = [irr(p, par, coupon_rate, months, freq, tax_rate) for p in quotes["price"]]
quotes.round({"irr": 4, "irr_after_tax": 4})

,price,irr,irr_after_tax
0,850000,-0.1503,-0.1537
1,900000,-0.1691,-0.1724
2,950000,-0.1869,-0.1901
3,1000000,-0.2036,-0.2067
4,1050000,-0.2194,-0.2224
5,1100000,-0.2344,-0.2374
6,1150000,-0.2486,-0.2516


## Excel

In [ ]:
pd.